# OpsMate QLoRA — the GPU enrichment track (optional)

This is the **one tier up** from the laptop lab. The core M6 lab runs a real LoRA fine-tune on CPU on an 8 GB machine — that is the load-bearing path and you do not need this notebook to complete the module. This notebook exists for when you want to feel **QLoRA**: LoRA trained on a **quantized** (4-bit) base, which is the technique that lets a much bigger model fit on a single small GPU.

**The idea (one beat):** QLoRA freezes the base in 4-bit (bitsandbytes NF4), so the big frozen weights take a quarter of the memory, and trains the small LoRA adapter in higher precision on top. Same removable-fittings idea as the lab; the only new thing is that the building itself is stored compressed while you renovate. It matters when the base is large enough that fp16 will not fit on your GPU — exactly the case this notebook shows and the CPU lab cannot.

**Honest availability note.** This needs a CUDA GPU. On **Google Colab**, `Runtime → Change runtime type → T4 GPU` (free tier, availability varies by the hour — you may be told none is free; try later or use a paid tier). `bitsandbytes` 4-bit is **CUDA-only** — it will not run on Colab CPU, on Apple Silicon, or on the course's laptop path. If you have no GPU, read this notebook for the shape and stay on the CPU lab; you are not missing any assessed content.

In [ ]:
# 0. Confirm we actually have a GPU — QLoRA needs CUDA. If this prints 'no CUDA',
#    switch the runtime to a GPU (Runtime -> Change runtime type -> T4 GPU) or
#    stay on the CPU lab. bitsandbytes 4-bit will not run without CUDA.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('no CUDA GPU — this notebook is the GPU track; use the CPU lab instead')

In [ ]:
# 1. Install the QLoRA stack. bitsandbytes is the 4-bit quantization backend that
#    makes the 'Q' in QLoRA — it is what the CPU lab cannot use.
!pip -q install "transformers>=4.51" peft accelerate bitsandbytes datasets trl

In [ ]:
# 2. Bring your synthetic data. Upload the train.jsonl that synthesize.py produced
#    on your laptop (labs/opsmate/train/train.jsonl) using the Colab file panel, or
#    regenerate it here against any OpenAI-compatible endpoint. The DATA is identical
#    to the CPU lab — only the training hardware changes.
import json, os
TRAIN = 'train.jsonl'
assert os.path.isfile(TRAIN), 'upload labs/opsmate/train/train.jsonl to the Colab session first'
rows = [json.loads(l) for l in open(TRAIN) if l.strip()]
print(len(rows), 'training samples')

In [ ]:
# 3. Load the base in 4-bit (NF4) — this is the QLoRA memory trick. The big frozen
#    weights live in 4-bit; the LoRA adapter trains in higher precision on top.
#    Swap BASE for a larger model (e.g. Qwen2.5-3B) to feel what QLoRA unlocks that
#    fp16 could not fit — the whole reason to reach for it.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

BASE = 'Qwen/Qwen3-0.6B'  # matches the lab base; bump this to a bigger model on a bigger GPU
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','v_proj'],
                  lora_dropout=0.0, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()  # same ~0.19% story — the base is just 4-bit now

In [ ]:
# 4. Train. Same chat-template rendering as the CPU lab (train and serve must share
#    the template). On a T4 this is seconds per epoch for a few hundred samples.
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def render(ex):
    text = tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return tok(text, truncation=True, max_length=512)

ds = Dataset.from_list(rows).map(render, remove_columns=['messages','source'])
args = TrainingArguments(output_dir='qlora-out', num_train_epochs=2,
                         per_device_train_batch_size=4, learning_rate=2e-4,
                         logging_steps=5, save_strategy='no', report_to=[])
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()
model.save_pretrained('qlora-adapter')
print('adapter saved to qlora-adapter/ — download it and merge/convert as in the CPU lab')

## Where this lands

The adapter this notebook trains is the **same kind of artifact** as the CPU lab's — a small `q_proj/v_proj` LoRA. You would download `qlora-adapter/`, then run the lab's `merge_and_convert.py` locally (merge is a CPU-friendly step) to produce a servable GGUF. QLoRA changed *how the base was held during training* (4-bit, on a GPU), not *what you ship*.

**When you would actually reach for this at work:** the base model you want to tune is too big to fit in fp16 on the GPU you have. On a single small GPU, QLoRA is often the difference between "can tune a 7B/13B" and "cannot tune it at all". On the tiny 0.6B model of this course it is overkill — which is exactly why the load-bearing lab stays on CPU and this notebook is enrichment.